# Extracting Data from PDFs

In this notebook, we'll look at how you can extract tabular data from a pdf using the base installation of [`img2table`](https://github.com/xavctn/img2table), a Python library that allows us to use the OCR engine [Tesseract](https://github.com/tesseract-ocr/tesseract).

In [ ]:
# Note: if installing locally, tesseract and poppler must be installed on your computer
%pip install pytesseract img2table

In [ ]:
import pandas as pd
from img2table.document import PDF
from img2table.ocr import TesseractOCR

# Set up Tesseract OCR engine
ocr_engine = TesseractOCR(lang="eng")

import requests

# The PDF file could be local, or it could be on the web.
# We're going to use 
# Lane, K., Pomeroy, E. & Davila, M. (2018). Over Rock and Under Stone: Carved Rocks and Subterranean Burials at Kipia, Ancash, AD 1000 – 1532. Open Archaeology, 4(1), pp. 299-321. Retrieved 20 Jul. 2018, from doi:10.1515/opar-2018-0018
url = 'https://researchonline.ljmu.ac.uk/id/eprint/8451/8/Rock%20and%20Under%20Stone%20Carved%20Rocks%20and%20Subterranean%20Burials%20at%20Kipia%20Ancash%20AD%201000%20%201532.pdf'

response = requests.get(url)
file_path = '../data/over-rock-under-stone.pdf'

if response.status_code == 200:
    with open(file_path, 'wb') as file:
        file.write(response.content)
    print('File downloaded successfully')
else:
    print('Failed to download file')


In [ ]:
# Now load the PDF, and let's grab the data in Table 1 which is on pg 6 of the pdf 
# (so we enter "5", since counting starts at 0 in most programming languages, including Python)
pdf = PDF(src='../data/over-rock-under-stone.pdf', pages=[5])

# The extract table command-- borderless_tables is important here since it helps detect tables without visible gridlines (like ours)
# min_confidence is saying that we want Tesseract to be at least 50% sure that what it's extracting is actually a table
extracted_tables = pdf.extract_tables(
    ocr=ocr_engine,
    implicit_rows=True,
    borderless_tables=True,
    min_confidence=50
)

# The output of extracted_tables is a dict
# Note how we can see the table description, and the bounding box coordinates that indicate where our table is on the page
extracted_tables.values()

In [ ]:
# We know our table is the first aka only table detected on page 6, so let's put it in a dataframe
page_tables = list(extracted_tables.values())[0]

table1_df = page_tables[0].df
table1_df

In [ ]:
# Looks like a table! But it needs to be cleaned up a bit
# For the purpose of our analysis, we can just use the first row as our headers and remove the second row
# aka the second row of headers, since the table in our PDF had the full headings split across two lines
table1_clean = table1_df.copy()
table1_clean.columns = table1_clean.iloc[0]
table1_clean = table1_clean.drop(table1_clean.index[0:2]).reset_index(drop=True)

table1_clean

In [ ]:
# Since we have a dataframe, let's plot something! 
import matplotlib.pyplot as plt
import re
 
# data is all strings, so we have to turn it into a numeric datatype
 
# first we look in the C14 column for things that match digits
table1_clean['years'] = table1_clean['C14'].str.extract(r'(\d{3})').astype(float)
 
# then we remove the % sign and convert to numeric
table1_clean['RegPerc'] = table1_clean['Pit'].str.extract(r"(\d+)").astype(float)
 
# now we can make a scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(table1_clean['Pit'], table1_clean['C14'], facecolors='none', edgecolors='black')
plt.xlabel('Pit')
plt.ylabel('C14')
plt.show()